In [1]:
import os 
import sys 
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
import seaborn as sns 
from analyzer import * 

sns.set_theme(
    style="whitegrid",
    context="paper",
) 

vv = VQAResults(results_path='/home/david/Desktop/yuna/HPA/evaluation/scored/humans/human_vqa_per_question.json')    
human_vqa = vv.human 
vm = vv.model 
vm = vm[vm['finetuned'] == 'Pretrained']  

   Loading VQA annotations from /home/david/Desktop/yuna/data/v2_mscoco_val2014_annotations.json...
   ✓ Loaded 214354 VQA annotations
[]
[]


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-mpnet-base-v2")
embeddings = model.encode(questions, normalize_embeddings=True)


In [79]:
import hdbscan

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=20,
    metric='euclidean'
)
labels = clusterer.fit_predict(embeddings)


AttributeError: module 'sklearn.metrics._dist_metrics' has no attribute 'DistanceMetric64'

In [74]:
questions= model_by_qtype.question_type.unique()

In [34]:
model_by_qtype = vm.groupby(['answer_type', 'question_type']).agg(
    correct_mean=('correct', 'mean'),
    question_id_nunique=('question_id', 'nunique')
).reset_index().sort_values(by='correct_mean', ascending=False)   

In [35]:
human_by_qtype = human_vqa.groupby(['answer_type', 'question_type']).agg(
    correct_mean=('correct', 'mean'),
    question_id_nunique=('question_id', 'nunique')
).reset_index().sort_values(by='correct_mean', ascending=False)

In [38]:
pd.merge(human_by_qtype, model_by_qtype, on=['answer_type', 'question_type', 'question_id_nunique'], suffixes=('_human', '_model')).to_csv('./tables/VQA_qtype_acc.csv', float_format="%.1f")

In [ ]:
vv.human.answer_type.unique() # , vv.model 

In [18]:
mm = MMStarResults()
md = mm.model_mc
md.finetuned.unique()  

array(['Pretrained', 'JS VQA (GT)', 'SFT-Blind MMStar (n=15)',
       'JS-Blind VQA (n=10)', 'JS-Blind VQA (n=15)',
       'JS-Blind MMStar (n=15)', 'SFT-Blind VQA', 'SFT-VQA',
       'SFT-Blind MMStar'], dtype=object)

In [44]:
md.category.unique()

array(['coarse perception', 'fine-grained perception',
       'instance reasoning', 'logical reasoning', 'science & technology'],
      dtype=object)

In [48]:
correct_lr_qids= md[(md['finetuned'] == 'Pretrained') & (md['category'] == 'logical reasoning') & (md['correct'] == 100)].question_id.unique()

In [70]:
pd.set_option("display.max_colwidth", None)
result = (
    df[df['condition'] == 'inst blind']
    .groupby(['model', 'condition'], as_index=False)
    .agg(correct=('correct', 'mean'),
         output=('output', 'first'))
    .sort_values(by='correct', ascending=False)
)

print(result.to_string(index=False))

             model  condition  correct                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  

In [ ]:
example = correct_lr_qids[0]
df = md[(md['finetuned'] == 'Pretrained') &  (md['question_id'] == example)].sort_values(['correct', 'condition'])
print(df.iloc[0]['question'], df.iloc[0]['answer'], '\n',
      # 'answer_type: ', df.iloc[0]['answer_type'],  '\n',
      # 'question_type: ', df.iloc[0]['question_type'],  
      )
df.pivot(index=['condition'], columns='model', values='output')   

Approximately what proportion of the picture is occupied by the bus in the image?
Options: A: 0.8, B: 1, C: 0.6, D: 0.3
Provide only the letter corresponding to the correct choice (A, B, C, or D).
Answer: C 



model,InternVL 3.5 (1B),InternVL 3.5 (2B),InternVL 3.5 (4B),InternVL 3.5 (8B),LLaVA‑1.5 (7B),LLaVA‑Mistral (7B),LLaVA‑Vicuna (7B),Qwen3 (4B),Qwen3 (8B),Qwen3 Base (8B),Qwen3‑VL (2B),Qwen3‑VL (4B),Qwen3‑VL (8B)
condition,,,,,,,,,,,,,
,C: 0.6,To determine the proportion of the picture occ...,C,C,A,C,A,NaN,NaN,To determine the proportion of the picture occ...,To determine the proportion of the picture occ...,The bus in the image occupies a significant po...,C: 0.6
blind,The image shows a bus and a person. The bus oc...,To determine the proportion of the picture occ...,D,C: 0.6,A,D,A,NaN,NaN,NaN,"Based on the provided image, which is complete...",B,D
inst blind,The correct answer is:\n\nA: 0.8\n\nThis is be...,To determine the proportion of the picture occ...,To determine the proportion of the picture occ...,D: 0.3,A,C: 0.6,A,"<think>\nOkay, so the question is asking appro...","<think>\nOkay, so the question is asking about...",NaN,"Based on the provided text, there is no image ...","Since no image is provided, we must imagine a ...","Since no image is provided, I must rely on the..."


In [20]:
md[(md['finetuned'] == 'Pretrained') & (md['category'] != 'science & technology')].pivot_table(index=['model'], columns=['category'], values=['correct'], 
    margins=True,
    margins_name='Average' ).to_latex('./tables/1_mmstar_blind.tex', float_format="%.1f")       
    
md[md['finetuned'] == 'Pretrained'].pivot_table(
    index=['model', "condition"], 
    columns=['category'],  
    values=['correct'],  
    aggfunc='count',  
) 

correct                          \
category                      coarse perception fine-grained perception   
model              condition                                              
InternVL 3.5 (1B)                     57.352941               29.702970   
                   blind              22.058824               31.683168   
                   inst blind         25.000000               29.702970   
InternVL 3.5 (2B)                     52.941176               47.524752   
                   blind              27.941176               23.762376   
                   inst blind         20.588235               25.742574   
InternVL 3.5 (4B)                     61.764706               61.386139   
                   blind              23.529412               30.693069   
                   inst blind         20.588235               25.742574   
InternVL 3.5 (8B)                     69.117647               61.386139   
                   blind              30.882353               32.673267   
                   inst blind         27.941176               32.673267   
LLaVA‑1.5 (7B)                        55.882353               25.742574   
                   blind              41.176471               25.742574   
                   inst blind         36.764706               27.722772   
LLaVA‑Mistral (7B)                    55.882353               34.653465   
                   blind              35.294118               29.702970   
                   inst blind         25.000000               24.752475   
LLaVA‑Vicuna (7B)                     57.352941               25.742574   
                   blind              33.823529               20.792079   
                   inst blind         32.352941               20.792079   
Qwen3 (4B)         inst blind         25.000000               33.663366   
Qwen3 (8B)         inst blind         17.647059               29.702970   
Qwen3 Base (8B)                       16.176471               20.792079   
Qwen3‑VL (2B)                         64.705882               54.455446   
                   blind              19.117647               23.762376   
                   inst blind         30.882353               28.712871   
Qwen3‑VL (4B)                         64.705882               60.396040   
                   blind              26.470588               29.702970   
                   inst blind         17.647059               34.653465   
Qwen3‑VL (8B)                         66.176471               66.336634   
                   blind              30.882353               25.742574   
                   inst blind         30.882353               30.693069   

                                                                    \
category                      instance reasoning logical reasoning   
model              condition                                         
InternVL 3.5 (1B)                      60.000000         50.000000   
                   blind               36.363636         22.727273   
                   inst blind          29.090909         27.272727   
InternVL 3.5 (2B)                      58.181818         72.727273   
                   blind               23.636364         31.818182   
                   inst blind          18.181818         36.363636   
InternVL 3.5 (4B)                      76.363636         68.181818   
                   blind               21.818182         36.363636   
                   inst blind          25.454545         31.818182   
InternVL 3.5 (8B)                      80.000000         81.818182   
                   blind               29.090909         50.000000   
                   inst blind          27.272727         63.636364   
LLaVA‑1.5 (7B)                         40.000000         36.363636   
                   blind               29.090909         22.727273   
                   inst blind          29.090909         27.272727   
LLaVA‑Mistral (7B)                     38.181818         40.909091   
         

In [10]:
vv.human['answer_similarity'] = vv.human['answer_similarity']*100

In [ ]:
from  utils.plot_histogram import * 

draw_histogram_embedding_similarity(vv.human['answer_similarity'], vm['answer_similarity'])
draw_ecdf_embedding_similarity(vv.human['answer_similarity'], vm['answer_similarity']) 

In [99]:
results = pd.merge(df.groupby(["category",	"l2_category"]).agg({'correct': 'mean', 'question':'nunique'}).reset_index(),
        blind.groupby(["category",	"l2_category"])['correct'].mean().reset_index(), 
        on=["category",	"l2_category"], suffixes=('_human', '_model')
        )
        
results['diff'] = np.abs(results['correct_model'] - results['correct_human'])
results.sort_values(by=['diff']) 

,category,l2_category,correct_human,question,correct_model,diff
8,instance reasoning,single-instance reasoning,0.343750,4,0.340909,0.002841
13,math,numeric commonsense and calculation,0.250000,1,0.242857,0.007143
10,logical reasoning,common reasoning,0.240385,13,0.232877,0.007508
3,fine-grained perception,localization,0.125000,1,0.083333,0.041667
2,coarse perception,image style & quality,0.250000,2,0.291667,0.041667
0,coarse perception,image emotion,0.375000,4,0.289474,0.085526
15,science & technology,biology & chemistry & physics,0.187500,2,0.100000,0.087500
5,fine-grained perception,recognition,0.321429,7,0.208333,0.113095
11,logical reasoning,diagram reasoning,0.166667,6,0.280488,0.113821
12,math,geometry,0.270000,25,0.390000,0.120000
